# On-Policy Distillation on SageMaker HyperPod EKS

Distil a **9B teacher** (`Qwen/Qwen3.5-9B`, bf16, sharded over TP=2) into a **2B student**
(`Qwen/Qwen3.5-2B`, instruct, LoRA r=128) by training on the student's *own* GSM8K rollouts with
**dense, per-token feedback**: the teacher scores every token the student samples, and the student
minimises the reverse KL. No verifier and no reward model, following the Thinking Machines
*On-Policy Distillation* recipe — and **no supervised warm-up**.

**Result** (all 1,319 GSM8K test problems, greedy): **74.8% → 84.0%** after 25 steps (~2.3 h),
closing **47%** of the gap to the teacher's 94.3% (paired sign test z = +7.45: 195 problems fixed,
73 broken). 50 steps give the same (83.9%).

**It starts with a baseline.** Part 1 evaluates the teacher and every candidate student on the same
1,319 problems before anything trains, and a comparison dashboard picks the student: the 2B leaves a
20-point gap to distil, the 0.8B (52.0%) leaves more but its capacity caps it at ~65%, and the 4B
(92.8%) is already within 2 points of the teacher.

**Everything runs on one `ml.g5.24xlarge`** (4× A10G), partitioned between the teacher (2 GPUs),
the trainer (1) and a vLLM rollout sampler (1). This notebook is only the **control panel** —
nothing trains in the kernel, so a CPU kernel is fine.

**Flow:** 0 Setup → 1 Baseline eval & model selection (teacher, then each candidate → comparison
dashboard) → 2 Sampler + distillation Job → 3 Monitor → 4 Evaluate the distilled student (dashboard
+ final comparison) → Results → 5 Cleanup.

## Code structure

```
code/
├── on_policy_distillation.ipynb       this notebook — drives every step from baseline to cleanup
├── README.md                          deployment guide: CloudFormation stack, Studio space, first run
├── architecture.md                    design notes: GPU partition, data flow, why each component exists
├── manifests/
│   ├── env_vars                       single source of truth for every setting (shipped = run17); rendered into the ConfigMap
│   ├── env_vars.local                 git-ignored overrides for account-specific values (CTX, S3_BUCKET)
│   ├── opd-config.yaml-template       ConfigMap `opd-config` — every pod reads its configuration from it
│   ├── teacher-vllm.yaml-template     teacher Deployment + Service: vLLM, bf16 Qwen3.5-9B, TP=2 (GPUs 0-1)
│   ├── student-sampler.yaml-template  student sampler: vLLM with LoRA hot-swap (GPU 3); also serves the baseline evals
│   ├── student-distill-job.yaml-template  training Job `opd-student`: preflight, then train_distill.py (GPU 2)
│   ├── student-eval-job.yaml-template Job `opd-student-eval`: baseline / after / teacher eval over HTTP (no GPU)
│   ├── sft-job.yaml-template          optional supervised warm-up Jobs (teacher solutions, then LoRA SFT)
│   └── LAUNCH.md                      step-by-step launch and troubleshooting reference
├── src/
│   ├── train_distill.py               the on-policy loop: sample, teacher-score, reverse-KL advantage, LoRA update
│   ├── distill_step.py                loss math: per-token reverse KL, whitening, PPO-clipped surrogate (+ optional top-k KL)
│   ├── sampler.py                     rollout generation (vLLM or HF) and LoRA export/hot-swap into vLLM
│   ├── teacher_client.py              scores token sequences against the teacher via vLLM echo log-probs
│   ├── rollout_buffer.py              bounded buffer of scored rollouts with staleness tracking
│   ├── data.py                        training prompt pool: dedup, eval-contamination filter, shuffling
│   ├── prompts.py                     task specs, prompt building, answer extraction and scoring (shared by train + eval)
│   ├── model_setup.py                 student loading (bf16 base + fp32 LoRA), optimizer, scheduler, save/merge
│   ├── preflight.py                   8 pre-launch checks: memory, tokenizer, log-probs, prompt format, GPU partition
│   ├── eval_student.py                Inspect AI eval: before / after / teacher, headroom recovered, dashboard payload
│   ├── gen_teacher_data.py            optional: teacher solutions for the training split (SFT data / prompt filter)
│   ├── sft_warmup.py                  optional: LoRA supervised warm-up on the teacher's correct solutions
│   ├── compare_ckpts.py               debug: tensor-by-tensor diff of checkpoints against the base model
│   └── merge_debug.py                 debug: re-merge a LoRA adapter in fp32 and check the merge kept the gain
└── experiments/
    ├── README.md                      every run since the recipe worked: what changed, what it scored, findings
    ├── configs/                       run13–run19 overrides on top of env_vars (run17 = the shipped defaults)
    ├── results/gsm8k_evals.csv        every evaluated checkpoint: accuracy, z, gap closed, format, length
    └── scripts/                       eval_steps.sh (eval checkpoints as they land), summarize.py, results_table.py
```

## Architecture — one node, four GPUs, partitioned

Everything is co-resident on a single `ml.g5.24xlarge` (4x A10G 24 GB). K8s allocates GPUs to a pod
**exclusively** — no MPS or time-slicing on this cluster — so the four devices are *partitioned*,
never shared. Oversubscribing them raises no error anywhere; the losing pod just sits `Pending`,
which is why `preflight.py` check **[8]** does the subtraction before anything launches — over
`TEACHER_TP × TEACHER_REPLICAS`, since each replica claims `TEACHER_TP` GPUs of its own. As
configured that is 2 + 1 + 1 = **4 of 4** — the node is exactly full.

| GPU | Occupant | Footprint |
|---|---|---|
| 0 | `teacher-vllm` rank 0 — half of one bf16 `Qwen/Qwen3.5-9B`, TP=2, `replicas: 1` | ~9.0 GB weights + ~7.8 GB cache |
| 1 | `teacher-vllm` rank 1 — the other half | ~9.0 GB weights + ~7.8 GB cache |
| 2 | `opd-student` — the trainer (bf16 2B + fp32 LoRA r=128) | peaks on 1024-token rollouts (`MICRO_BATCH_SIZE=2`) |
| 3 | `student-sampler` — vLLM rollouts, LoRA hot-swap; serves each baseline candidate in Part 1 | ~4 GB weights + KV |

**One teacher sharded over TP=2, unquantised.** Teacher scoring is a single prefill forward pass
per request (echo-logprobs, zero generated tokens), so its per-request cost is fixed by the model
size — sharding does not *buy* throughput, but it is what lets a bf16 9B run on one A10G at all
(18.0 GiB does not fit against a ~20.2 GiB per-GPU budget). At TP=2 each rank holds ~9.0 GiB and
~7.8 GiB/GPU is left for cache. TP=2 revives a PCIe all-reduce (A10G has no NVLink), and *which*
pair of GPUs the kubelet assigns matters: GPUs 0-1 and 2-3 sit under separate PCIe switches on
g5.24xlarge, so a split pair is measurably slower. `preflight` check [4] warns on `SYS` pairings.

**The eval needs no GPU.** It generates the student's answers through the sampler (the base
candidate in the Part 1 baseline, the trained adapter afterwards) and the teacher's through `teacher-vllm`, so `opd-student-eval` is an HTTP
client and runs alongside the trainer even though the node is full.

**Why bf16 rather than a 4-bit teacher at TP=1.** The teacher's per-token log-probs *are* the
supervision signal, so quantisation error lands directly in the KL target. W4A16 of a 9B perturbs
that signal more than the 27B's MLP-only W4A16 did, and every 4-bit 9B checkpoint on the Hub is a
third-party build. bf16 at TP=2 gives exact log-probs at the cost of a second GPU and an
all-reduce. `preflight.py` check **[2]** hard-fails bf16 at TP=1 by name (`0.90 × 22.5 GiB` per-GPU
budget minus 18.0 GiB weights leaves negative room for the cache). The W4A16 TP=1 fallback
(`sanskar003/Qwen3.5-9B-AWQ`, `TEACHER_QUANTIZATION=compressed-tensors`) is documented in
`manifests/env_vars` if you need the spare GPU back — accept the log-prob perturbation.

**The cache arithmetic is hybrid-aware.** Qwen3.5 alternates 3 `linear_attention` layers to 1
`full_attention`, so of 32 layers only **8** hold a growing KV cache (32 KiB/token) while **24**
hold a fixed fp32 recurrent state **per sequence** (48 MB). At TP=2 both terms are sharded, so
each rank's ~7.8 GiB of cache holds ~253k tokens of KV, or ~141 concurrent sequences at the full
`max_model_len`; `TEACHER_MAX_NUM_SEQS=48` is **per replica** and well inside that — memory has
stopped being the binding constraint, prefill compute is.

```mermaid
flowchart LR
  subgraph NODE["ONE ml.g5.24xlarge · 4x A10G · GPUs partitioned exclusively"]
    subgraph TP["GPU 0-1 — teacher (inference only), sharded TP=2"]
      T1["teacher-vllm (GPUs 0-1)<br/>Qwen3.5-9B, bf16, TP=2<br/>"]
      SVC(["Service teacher-vllm:8000"])
      T1 --> SVC
    end
    subgraph SP["GPU 2 — student"]
      S["Job opd-student<br/>train_distill.py<br/>LoRA: bf16 base + fp32 adapter"]
    end
    subgraph SA["GPU 3"]
      R["Deployment student-sampler<br/>vLLM rollout generation"]
    end
  end
  E["Job opd-student-eval<br/>no GPU — HTTP to sampler + teacher"]
  subgraph STO["SHARED STORAGE"]
    FSX[("FSx /fsx<br/>HF cache · checkpoints · metrics · eval")]
    S3[("S3 /s3<br/>models · code")]
  end
  S -- "1 . prompts" --> R
  R -- "2 . rollout token_ids" --> S
  S -- "3 . student token_ids" --> SVC
  SVC -- "4 . per-token teacher logprobs" --> S
  S -- "LoRA hot-swap every SAMPLER_SYNC_EVERY steps" --> R
  S -. write .-> FSX
  E -. read .-> FSX
  E -- "teacher phase" --> SVC
  E -- "baseline (before) / after phases" --> R
  S -. read .-> S3
  T1 -. read .-> S3
```

**Per-step loop:** (1) the sampler generates 4 rollouts for each of 64 prompts from the student's
current policy → (2) the teacher scores every sampled token via vLLM echo log-probs → (3)
`reverse_kl = log π_student − log π_teacher`, clamped to `±RKL_CLIP`, negated and whitened into a
per-token advantage, with zero credit on the final-answer line → (4) PPO-clipped importance-sampled
loss, LoRA update, and the new adapter is hot-swapped into the sampler. The teacher never trains.

## Part 0 — Setup

Download and install kubectl

In [ ]:
%%bash --out bash_env

HP_CLUSTER_NAME=<your-hyperpod-cluster-name>   # edit me
EKS_CLUSTER_ARN=`aws sagemaker describe-cluster --cluster-name $HP_CLUSTER_NAME --query "Orchestrator.Eks.ClusterArn" --output text`
EKS_CLUSTER_NAME=`echo $EKS_CLUSTER_ARN|awk -F / '{print $2}'`
K8S_VERSION=`aws eks describe-cluster --name $EKS_CLUSTER_NAME --query "cluster.version" --output text`
S3_URL="https://docs.aws.amazon.com/eks/latest/userguide/install-kubectl.html#linux_amd64_kubectl"
KUBECTL_PATH=$(curl -s "${S3_URL}/"| grep -Po 'https://[^<]*'"${K8S_VERSION}"'[^<]*linux/amd64/kubectl()?') 
KUBECTL_PATH=`echo ${KUBECTL_PATH}| cut -d ' ' -f 1`
#Download and install kubectl
curl -O $KUBECTL_PATH
mkdir -p ~/bin
chmod +x kubectl
mv kubectl ~/bin
echo $EKS_CLUSTER_ARN

Point `kubectl` at the cluster and define a small `sh()` shell helper. (`kubectl` lives in `~/bin`,
so we add it to PATH and call it by absolute path.)

In [ ]:
!pip install -q matplotlib 2>/dev/null

In [ ]:
import os, json, subprocess
os.environ["PATH"] = os.path.expanduser("~/bin") + os.pathsep + "/usr/local/bin" + os.pathsep + os.environ.get("PATH", "")

KUBECTL   = os.path.expanduser("~/bin/kubectl")
CTX       = bash_env.strip()
NAMESPACE = "default"
KCTL      = f"{KUBECTL} --context {CTX} -n {NAMESPACE}"
RUN_DIR   = "/fsx/opd/gsm8k-opd"     # must match RUN_DIR in manifests/env_vars (checkpoints + metrics on FSx)


def sh(cmd):
    r = subprocess.run(cmd, shell=True, executable="/bin/bash", capture_output=True, text=True)
    print(r.stdout or r.stderr); return r
def sh_out(cmd):
    return subprocess.run(cmd, shell=True, executable="/bin/bash", capture_output=True, text=True).stdout

sh("which kubectl aws envsubst")                              # sanity: tools resolve
sh(f"{KCTL} get nodes -L node.kubernetes.io/instance-type")  # the GPU pools
sh(f"{KCTL} get pods -o wide")                               # what's already running


#### Get the S3 bucket name (behind the `s3-claim` PV) and the EKS cluster ARN


In [ ]:
#EKS Cluster ARN
print(f"EKS Cluster ARN: {bash_env.strip()}")

In [ ]:
%%bash
#S3 Bucket - s3-claim
kubectl get pv $(kubectl get pvc s3-claim -o jsonpath='{.spec.volumeName}') -o jsonpath='{"Bucket: "}{.spec.csi.volumeAttributes.bucketName}{"\n"}'

**Note:** put both values in `manifests/env_vars.local` (git-ignored; sourced at the end of
`manifests/env_vars`), so account-specific values never get committed:

```bash
export CTX=<EKS cluster ARN printed above>
export S3_BUCKET=<bucket printed above>
```


#### Helpers used by every part

Every manifest is rendered from `manifests/env_vars`; `render()` applies one with per-call
overrides on top (used by the baseline loop to point the sampler and the eval at each candidate).
The eval Job reads its settings from the ConfigMap, so `run_eval()` re-renders the ConfigMap with
the same overrides before submitting it. Results on `/fsx` are read through the teacher pod, which
mounts it.

In [ ]:
import shlex, time

def env_var(name, default=""):
    # One value from manifests/env_vars -- the same file every manifest is rendered from.
    out = sh_out(f"set -a && source manifests/env_vars && set +a && printf '%s' \"${name}\"").strip()
    return out or default

def render(template, **overrides):
    # envsubst manifests/<template> from env_vars plus overrides, and apply it.
    exports = "".join(f"export {k}={shlex.quote(str(v))} && " for k, v in overrides.items())
    return sh(f"set -a && source manifests/env_vars && {exports}set +a && "
              f"envsubst < manifests/{template} | {KUBECTL} --context {CTX} apply -f -")

def wait_job(name, timeout=3600, poll=30):
    # Block until a Job completes or fails; returns True on success.
    t0 = time.time()
    while time.time() - t0 < timeout:
        state = sh_out(f"{KCTL} get job {name} -o jsonpath='{{.status.conditions[?(@.status==\"True\")].type}}'")
        if "Complete" in state:
            return True
        if "Failed" in state:
            print(f"{name} FAILED:"); sh(f"{KCTL} logs job/{name} --tail=30"); return False
        time.sleep(poll)
    print(f"{name}: still running after {timeout}s"); return False

def run_eval(**overrides):
    # One eval Job (Jobs are immutable, so delete and recreate) with overrides such as EVAL_PHASE,
    # EVAL_DIR, EVAL_BASE_MODEL, EVAL_FT_MODEL_DIR. The ConfigMap carries them into the pod.
    render("opd-config.yaml-template", **overrides)
    sh(f"{KCTL} delete job opd-student-eval --ignore-not-found --wait=true")
    render("student-eval-job.yaml-template", **overrides)
    ok = wait_job("opd-student-eval")
    sh(f"{KCTL} logs job/opd-student-eval --tail=-1 | grep -E '^=== \\[(before|after|teacher)\\] acc' || true")
    return ok

def fsx_cat(path):
    return sh_out(f"{KCTL} exec deploy/teacher-vllm -c vllm -- cat {path}")

def fsx_write(path, text):
    subprocess.run(f"{KCTL} exec -i deploy/teacher-vllm -c vllm -- sh -c "
                   f"{shlex.quote(f'mkdir -p $(dirname {path}) && cat > {path}')}",
                   shell=True, input=text, text=True, check=True)

def load_eval(eval_dir):
    # summary.json plus every phase's per-sample rows -- the dashboards' input. None if absent.
    raw = fsx_cat(f"{eval_dir}/summary.json")
    if not raw.strip():
        return None
    payload = json.loads(raw)
    payload["samples"] = {}
    for phase, res in payload["results"].items():
        rows = fsx_cat(res.get("samples_file") or f"{eval_dir}/samples_{phase}.jsonl")
        payload["samples"][phase] = [json.loads(l) for l in rows.splitlines() if l.strip()]
    return payload

TASK    = env_var("TASK")
RUN_DIR = env_var("RUN_DIR", RUN_DIR)
print(f"task={TASK} · student={env_var('STUDENT_MODEL')} (LoRA r={env_var('LORA_R')}) · "
      f"teacher={env_var('TEACHER_MODEL_ID')} · RUN_DIR={RUN_DIR}")

## Part 1 — Baseline eval and model selection

**Measure before you train.** Every model is evaluated on the same 1,319 GSM8K test problems
(4-shot, greedy, `#### N` answer match) before anything is distilled: the **teacher** is the
ceiling, each **candidate student** is a floor, and the distance between them is all that
distillation can recover. That decides which student to train.

| step | what | GPU | time |
|---|---|---|---|
| 1a | deploy the teacher (vLLM, TP=2) — it serves the teacher eval now and scores rollouts in Part 2 | 0-1 | ~10 min cold |
| 1b | eval the **teacher** once, then each **candidate**: the sampler serves one student at a time | 3 | ~6 min each + ~3 min per sampler swap |
| 1c | **comparison dashboard** → pick the student, set `STUDENT_MODEL` in `env_vars` | — | — |

The eval itself needs no GPU: every phase is an HTTP client of vLLM (the sampler for students, the
teacher's Service for the teacher).

### 1a — Deploy the teacher (vLLM, TP=2, one replica, GPUs 0-1)

A vLLM `Deployment` (**`replicas: 1`**, one teacher sharded across two GPUs) plus a `Service` in
front of it, serving **`Qwen/Qwen3.5-9B`** in bf16 (unquantised — `TEACHER_QUANTIZATION` is empty,
so `TEACHER_QUANT_FLAG` expands to nothing, and `TEACHER_DIST_FLAG=--distributed-executor-backend
mp` is added because `TEACHER_TP>1`). It exposes an OpenAI-compatible API, answers the baseline's
`teacher` phase, and during training returns per-token log-probs (echo-logprobs) — the only thing
the student needs from it.

**Why sharded over TP=2 instead of one whole copy on one GPU?** bf16 `Qwen/Qwen3.5-9B` is 18.0 GiB
and does not fit one A10G against a ~20.2 GiB budget. Sharded over two GPUs it becomes
~9.0 GiB/rank with ~7.8 GiB/GPU left for cache. The cost is a PCIe all-reduce per prefill layer
(A10G has no NVLink) and the fact that *which* pair of GPUs the kubelet picks matters — GPUs 0-1
and 2-3 sit under separate PCIe switches, so a split pair is measurably slower; `preflight` [4]
warns on `SYS` pairings. The W4A16 TP=1 fallback (`sanskar003/Qwen3.5-9B-AWQ`) is in `env_vars` if
you need the spare GPU back — accept the log-prob perturbation.

> `preflight.py` [2] verifies both the quantisation method (against the vLLM build's registry) *and*
> the scheme's minimum compute capability against the GPU, before you pull.

Three flags are not optional:

- **`--max-logprobs ${TEACHER_MAX_LOGPROBS}`** — without it vLLM returns `null` for echo log-probs.
  That was the original ~6-point result: the client turned nulls into `0.0`, the advantage collapsed
  to `−log π_student` (pure self-reinforcement), and `teacher_kl` still printed a convincing falling
  curve. `teacher_client.py` now **raises** instead of zeroing, and `preflight.py` check [5] catches
  it before a single training step. 1 is enough for the shipped loss; the experimental
  `TOPK_KL=k` needs `k`.
- **`--tensor-parallel-size 2 --distributed-executor-backend mp`** with `replicas: 1` — two GPUs
  in total for the teacher, not one and not four: the other two belong to the student and the
  sampler and the node is exactly full. Raising `TEACHER_REPLICAS` to 2 requires either dropping
  the sampler or switching to the W4A16 TP=1 fallback so the arithmetic fits.
- **`--served-model-name ${TEACHER_MODEL_ID}`** — clients must request this exact string. It stays
  a real HF id (`Qwen/Qwen3.5-9B`); `TEACHER_MODEL`, the thing vLLM loads, can point at
  `/s3/...` or `/fsx/...` for fast cold starts.

Both Deployments use `strategy: Recreate`: with every GPU claimed, a rolling update's surge pod
could never schedule, so a changed spec would wait forever.

> **Expect several minutes.** 18.0 GiB of bf16 weights are read across two GPUs and per-rank
> CUDA-graph capture runs on top of an NCCL init for the TP all-reduce; a cold first start is
> I/O-bound. The probes allow ~20 minutes. One pod stuck `Pending` means the partition is
> oversubscribed.

In [ ]:
# The ConfigMap must exist first -- every Job reads its whole configuration from it and passes
# NO arguments, so a stale ConfigMap means a pod running with a value nobody set.
render("opd-config.yaml-template")
render("teacher-vllm.yaml-template")
# A cold cache makes the first start I/O-bound; the probes allow ~20 min.
sh(f"{KCTL} rollout status deploy/teacher-vllm --timeout=1800s")   # returns once /v1/models is ready

# Expect ONE pod Running on two GPUs (TP=2 x 1 replica). Pending means the partition is
# oversubscribed: TEACHER_TP x TEACHER_REPLICAS + GPU_PER_NODE + SAMPLER_GPUS > NODE_GPUS.
sh(f"{KCTL} get pods -l app=teacher-vllm -o wide")

# GPUs 0-1 and 2-3 sit under separate PCIe switches; a split pair costs a slower all-reduce.
sh(f"{KCTL} exec deploy/teacher-vllm -c vllm -- nvidia-smi topo -m")

# The eval and the trainer load their code from $CODE_DIR (/s3/opd/src).
sh("set -a && source manifests/env_vars && set +a && "
   "aws s3 sync src s3://$S3_BUCKET/opd/src --exclude '__pycache__/*'")

### 1b — Baseline eval: the teacher, then every candidate student

The teacher runs first (`EVAL_PHASE=teacher`, through its Service). The student sampler (GPU 3)
serves one model at a time, so the loop then swaps it per candidate: re-render
`student-sampler` with `STUDENT_MODEL=<candidate>` (`Recreate` frees GPU 3 first), wait for it,
and run `EVAL_PHASE=before` into `/fsx/opd/baseline-<task>-<model>`. Each result is a normal eval
directory (`summary.json` + `samples_<phase>.jsonl`), so Part 4 reuses the chosen student's and the
teacher's as the `before` and `teacher` phases instead of re-running them.

The overrides apply to this cell only; `env_vars` stays the source of truth for training. Add
`"Qwen/Qwen3.5-4B"` to compare a third student (it scores 92.8% — too close to the teacher to be
worth distilling).

In [ ]:
TEACHER_ID = env_var("TEACHER_MODEL_ID")
CANDIDATES = ["Qwen/Qwen3.5-0.8B", "Qwen/Qwen3.5-2B"]     # e.g. + "Qwen/Qwen3.5-4B"

def baseline_dir(model):
    return f"/fsx/opd/baseline-{TASK}-{model.split('/')[-1].lower()}"

# 1) the teacher -- the ceiling. No sampler needed.
run_eval(EVAL_PHASE="teacher", EVAL_DIR=baseline_dir(TEACHER_ID))

# 2) each candidate student -- point the sampler at it, then eval its base model
for model in CANDIDATES:
    ov = dict(STUDENT_MODEL=model, EVAL_BASE_MODEL=model, EVAL_PHASE="before",
              EVAL_DIR=baseline_dir(model))
    render("student-sampler.yaml-template", **ov)
    sh(f"{KCTL} rollout status deploy/student-sampler --timeout=1200s")
    run_eval(**ov)

# The sampler is left serving the last candidate. Part 2 re-renders it from env_vars.

### 1c — Comparison dashboard: which student to distil

Four panels over the baseline directories: **accuracy** (± standard error) with the teacher as the
ceiling, the **teacher-solved problems each student misses** (the distillable headroom, counted on
the same problems), **answer-format compliance**, and **response length**. The same function draws
the final comparison in Part 4, with the distilled checkpoints added.

**Selection rule:** the strongest candidate that still leaves a gap well beyond noise
(`MIN_GAP_PP`, 10 points ≈ 7 standard errors) — a strong base keeps headroom-recovery high, and
the gap keeps the gain measurable.

| model | role | accuracy | gap to teacher | teacher-solved problems missed |
|---|---|---|---|---|
| `Qwen/Qwen3.5-0.8B` | candidate | 52.0% | 42.3 pp | 567 |
| **`Qwen/Qwen3.5-2B`** | **candidate → chosen** | **74.8%** | **19.6 pp** | **276** |
| `Qwen/Qwen3.5-4B` | optional candidate | 92.8% | 1.5 pp | 52 |
| `Qwen/Qwen3.5-9B` | teacher | 94.3% | — | — |

**Why not the 0.8B, with twice the gap?** Its capacity caps it: every 0.8B run (more steps, higher
LR) plateaued at 64–66%, ~30% of its gap, while the 2B closes 47% and ends 18 points higher. The
4B's gap is within a few standard errors of zero.

In [ ]:
import matplotlib.pyplot as plt

MIN_GAP_PP = 10.0
ROLE_COLORS = {"teacher": "#232f3e", "candidate": "#9aa0a6", "chosen": "#0f766e", "distilled": "#ff9900"}


def phase_entry(eval_dir, phase, label, role):
    # (label, role, aggregate result, per-sample rows) for one phase of one eval directory.
    p = load_eval(eval_dir)
    if not p or phase not in p["results"]:
        print(f"missing: {label} ({eval_dir}, phase {phase})"); return None
    return dict(label=label, role=role, res=p["results"][phase], rows=p["samples"].get(phase, []))


def compare_dashboard(entries, title):
    entries = [e for e in entries if e]
    teacher = next((e for e in entries if e["role"] == "teacher"), None)
    t_ok = {r["id"] for r in teacher["rows"] if r["correct"]} if teacher else set()
    labels = [e["label"] for e in entries]
    colors = [ROLE_COLORS[e["role"]] for e in entries]
    fig, ax = plt.subplots(2, 2, figsize=(15, 8.5))
    fig.suptitle(title, fontsize=13)

    # 1 -- accuracy +- stderr, teacher = ceiling
    a = ax[0, 0]
    accs = [e["res"]["accuracy"] for e in entries]
    errs = [e["res"].get("stderr") or 0 for e in entries]
    bars = a.bar(labels, accs, yerr=errs, capsize=6, color=colors)
    for b, acc, err in zip(bars, accs, errs):
        a.text(b.get_x() + b.get_width() / 2, acc + err + 0.01, f"{acc:.1%}", ha="center",
               va="bottom", fontweight="bold")
    if teacher:
        a.axhline(teacher["res"]["accuracy"], ls="--", lw=1.2, color="#232f3e", alpha=0.7)
    a.set_ylim(0, 1.12); a.set_title("Accuracy (dashed = teacher ceiling)"); a.grid(axis="y", alpha=0.3)

    # 2 -- distillable headroom: problems the teacher solves that this model misses
    a = ax[0, 1]
    missed = [sum(1 for r in e["rows"] if r["id"] in t_ok and not r["correct"]) for e in entries]
    bars = a.bar(labels, missed, color=colors)
    for b, m in zip(bars, missed):
        a.text(b.get_x() + b.get_width() / 2, m, str(m), ha="center", va="bottom")
    a.set_title("Teacher-solved problems the model misses"); a.set_ylabel("problems")
    a.grid(axis="y", alpha=0.3)

    # 3 -- answer-format compliance
    a = ax[1, 0]
    fmt = [e["res"].get("format_rate") or 0 for e in entries]
    a.bar(labels, fmt, color=colors)
    a.set_ylim(min(fmt + [0.9]) - 0.05, 1.0); a.set_title("Writes the answer marker (#### N)")
    a.grid(axis="y", alpha=0.3)

    # 4 -- response length
    a = ax[1, 1]
    data = [[r["output_tokens"] for r in e["rows"] if r.get("output_tokens")] for e in entries]
    bp = a.boxplot(data, patch_artist=True, showfliers=False, medianprops={"color": "#16191f"})
    for patch, c in zip(bp["boxes"], colors):
        patch.set_facecolor(c); patch.set_alpha(0.75)
    a.set_xticks(range(1, len(labels) + 1)); a.set_xticklabels(labels)
    a.set_title("Response length (output tokens)"); a.grid(axis="y", alpha=0.3)
    for a in ax.flat:
        a.tick_params(axis="x", labelsize=9)
    plt.tight_layout(); plt.show()

    # the same numbers as a table
    t_acc = teacher["res"]["accuracy"] if teacher else None
    print(f"{'model':<26}{'role':<11}{'accuracy':>9}{'gap (pp)':>10}{'missed':>8}{'tokens':>8}")
    for e, m in zip(entries, missed):
        gap = f"{(t_acc - e['res']['accuracy']) * 100:.1f}" if t_acc and e["role"] != "teacher" else "-"
        print(f"{e['label'].replace(chr(10), ' '):<26}{e['role']:<11}{e['res']['accuracy']:>9.1%}{gap:>10}{m:>8}"
              f"{e['res'].get('mean_output_tokens') or 0:>8.0f}")
    return missed


def short(model):
    return model.split("/")[-1].replace("Qwen3.5-", "")


entries = [phase_entry(baseline_dir(m), "before", f"{short(m)}\nbase", "candidate") for m in CANDIDATES]
entries.append(phase_entry(baseline_dir(TEACHER_ID), "teacher", f"{short(TEACHER_ID)}\nteacher", "teacher"))
compare_dashboard(entries, f"Baseline — {TASK}, all candidates and the teacher on the same problems")

# selection: the strongest candidate whose gap to the teacher is still >= MIN_GAP_PP
t_acc = entries[-1]["res"]["accuracy"] if entries[-1] else None
viable = [(e["res"]["accuracy"], m) for m, e in zip(CANDIDATES, entries)
          if e and t_acc and (t_acc - e["res"]["accuracy"]) * 100 >= MIN_GAP_PP]
if viable:
    print(f"\n=> distil {max(viable)[1]}  (set STUDENT_MODEL in manifests/env_vars; shipped: {env_var('STUDENT_MODEL')})")

## Part 2 — Launch the sampler, then the student (on-policy distillation Job, GPUs 2-3)

From here on `manifests/env_vars` is the configuration — the shipped values are run17's
(`experiments/configs/run17.env`): **`Qwen/Qwen3.5-2B`**, LoRA **r=128** (α=256), `LR=2e-5`,
`STEPS=50` (the plateau is reached by step 25), 64 prompts × 4 samples per step at temperature 0.7,
whitened advantages, answer line masked, `MICRO_BATCH_SIZE=2`.

The trainer calls `sampler.generate()` at **step 0**, so the vLLM student sampler (GPU 3)
must already be Running before the trainer Job starts — otherwise the first rollout fails
with connection refused and the pod exits. The next cell therefore: (1) re-renders
`student-sampler` from `env_vars` — the chosen student, with `--max-lora-rank ${LORA_R}` — and
waits for it; (2) re-applies the ConfigMap (Part 1 left eval overrides in it); (3) submits
`opd-student`. The Job also waits for `${STUDENT_SAMPLER_URL}/models` internally before running
`preflight.py`.

Once launched, the trainer waits for the teacher `Service`, runs `preflight.py`, then
`torchrun train_distill.py`: generate rollouts → have the teacher score them → per-token
reverse-KL advantage → clipped IS loss → update the LoRA adapter. All on its own GPU,
inside the cluster.

**Two gates run before any real work**, both designed to fail loudly instead of wasting hours:

- `preflight.py` — 8 checks. TP divisibility, teacher memory fit at TP with the resulting KV
  headroom, a **shared tokenizer** (the teacher scores raw token ids, so a differing vocabulary
  means it scores different text — silent and fatal), GPU topology, **echo-logprobs actually
  populated**, student memory fit, train/eval prompt-format agreement, and the single-node GPU
  partition.
- a **step-0 sanity gate** in the trainer — it prints the measured teacher and student log-probs and
  exits `3` if the reverse KL is not positive, rather than spend hours optimising against noise.

**If the run dies part-way through**, do not restart from zero: set `INIT_ADAPTER` to the newest
`${RUN_DIR}/student-step<N>` checkpoint (written every `SAVE_EVERY` steps), lower `STEPS` to the
remaining count, re-apply the ConfigMap, then delete and recreate the Job.

In [ ]:
# 1) the sampler, serving the chosen student from env_vars (Recreate replaces Part 1's last candidate)
render("student-sampler.yaml-template")
sh(f"{KCTL} rollout status deploy/student-sampler --timeout=1200s")

# 2) code + a clean ConfigMap (the baseline loop left eval overrides in it)
sh("set -a && source manifests/env_vars && set +a && "
   "aws s3 sync src s3://$S3_BUCKET/opd/src --exclude '__pycache__/*'")
render("opd-config.yaml-template")

# 3) the training Job (immutable: delete any previous run's first)
sh(f"{KCTL} delete job opd-student --ignore-not-found --wait=true")
render("student-distill-job.yaml-template")

# 4) the node: teacher-vllm (GPUs 0-1) + opd-student (GPU 2) + student-sampler (GPU 3)
sh(f"{KCTL} get pods -o wide")

### The vLLM rollout sampler (GPU 3)

The loop spends most of its wall clock **generating** rollouts, and HF `generate` is roughly 5–10×
slower than vLLM for this. GPU 3 is exactly what the teacher's 2 and the trainer's 1 leave over. In
Part 1 the same Deployment served each baseline candidate; here it serves the student being trained.

The trainer hot-swaps its LoRA adapter into the sampler every `SAMPLER_SYNC_EVERY` (1) steps via
`POST /v1/load_lora_adapter`. Two details make that work:

- **Adapter key renaming.** vLLM serves Qwen3.5 as the multimodal `Qwen3_5ForConditionalGeneration`,
  whose language layers live under `model.language_model.layers.*`. A PEFT adapter saved from the
  text-only model uses `model.layers.*`, and vLLM **silently ignores it** — every rollout would come
  from the base student. `sampler.export_adapter_for_vllm` writes a renamed copy, and the trainer
  checks at the 5th sync that the adapter really changes the sampler's log-probs.
- **Unique adapter names per process**, so a restarted trainer never collides with names a previous
  attempt left registered in the sampler.

`GROUP_SIZE` doubles as the sampler's `--max-num-seqs` and `LORA_R` as its `--max-lora-rank`, so
re-apply the sampler after changing either. To skip the sampler, set `STUDENT_SAMPLER_URL=` and
`SAMPLER_GPUS=0`: in-process sampling is correct, just slower.

## Part 3 — Monitor convergence

The plots read `${RUN_DIR}/metrics_rank0.jsonl` on FSx through the teacher pod (it shares the `/fsx`
mount). Re-run these cells to refresh while training runs (~5.5 min per step for the 2B).

**`teacher_kl` is confounded by length.** Long, rambling text is easy for the teacher to predict,
so a student drifting toward longer answers *lowers* `teacher_kl` without learning anything. Read it
together with `truncated_frac` and `completion_tokens_mean`, and judge progress by the eval.

| metric | healthy | what it means when it isn't |
|---|---|---|
| `teacher_kl` | slowly falling at stable length | flat ⇒ no learning; falling *while length rises* ⇒ drift, not learning |
| `truncated_frac` | stable or falling (~10–25%) | climbing fast ⇒ collapse into non-answers; lower `LR` |
| `answer_masked_rollouts` | most of the 256 rollouts | falling ⇒ the student is stopping writing final answers |
| `clip_frac` | < ~0.2 | high ⇒ trainer/sampler mismatch or stale rollouts |
| `grad_norm` | near/under `GRAD_CLIP` | pinned at it ⇒ `LR` too high |
| `sec_gen` / `sec_score` / `sec_train` | — | where the wall clock goes |

In [ ]:
sh(f"{KCTL} get pods -o wide")
sh(f"{KCTL} logs job/opd-student --tail=40")     # per-step teacher_kl; re-run to refresh

In [ ]:
import matplotlib.pyplot as plt

tpod = sh_out(f"{KCTL} get pods -l app=teacher-vllm -o jsonpath='{{.items[0].metadata.name}}'").strip()
raw  = sh_out(f"{KCTL} exec {tpod} -c vllm -- cat {RUN_DIR}/metrics_rank0.jsonl") if tpod else ""
rows = [json.loads(l) for l in raw.splitlines() if l.strip()]

if not rows:
    print("No metrics yet -- the Job is still starting (image pull, teacher wait, preflight). Re-run shortly.")
else:
    steps = [r["step"] for r in rows]
    fig, ax = plt.subplots(1, 3, figsize=(16, 3.8))

    # 1 -- the training signal (read it together with length: see the note above)
    ax[0].plot(steps, [r["teacher_kl"] for r in rows], color="#232f3e")
    ax[0].set_title("teacher_kl (mean reverse KL)")

    # 2 -- answer behaviour: capped rollouts and rollouts that still write an answer line
    ax[1].plot(steps, [r["truncated_frac"] for r in rows], color="#d93025", label="capped fraction")
    if "answer_masked_rollouts" in rows[0]:
        ax[1].plot(steps, [r["answer_masked_rollouts"] / r["batch"] for r in rows], color="#1e8e3e",
                   label="writes answer line")
    ax[1].set_ylim(0, 1); ax[1].set_title("Rollout health"); ax[1].legend(fontsize=8)

    # 3 -- rollout length and wall clock
    ax[2].plot(steps, [r["completion_tokens_mean"] for r in rows], color="#ff9900", label="mean tokens")
    ax2 = ax[2].twinx()
    ax2.plot(steps, [r["sec_gen"] + r["sec_score"] + r["sec_train"] for r in rows], color="#9aa0a6", label="sec/step")
    ax[2].set_title("Length (orange) and seconds per step (grey)")

    for a in ax:
        a.set_xlabel("step"); a.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

    last = rows[-1]
    print(f"{len(rows)} steps · teacher_kl={last['teacher_kl']:.4f} · capped={last['truncated_frac']:.3f} · "
          f"tokens={last['completion_tokens_mean']:.0f} · clip_frac={last.get('clip_frac', 0):.4f} · "
          f"grad_norm={last.get('grad_norm', 0):.3f}")


## Part 4 — Evaluate the distilled student (Inspect AI)

`src/eval_student.py` runs the same task as the baseline — **all 1,319 GSM8K test problems**,
4-shot (matching `TRAIN_FEWSHOT`; preflight [7] enforces it), greedy, `#### N` answer match:

| phase | model | source |
|---|---|---|
| `before` | `Qwen/Qwen3.5-2B` | **carried forward** from the Part 1 baseline |
| `teacher` | `Qwen/Qwen3.5-9B` | **carried forward** from the Part 1 baseline |
| `after`  | `student-step<N>` | a LoRA checkpoint, loaded into the sampler at eval time |

Only `after` runs (~6 min per checkpoint, no GPU, alongside training if it is still going): the
cell writes the baseline's `before` and `teacher` results into each checkpoint's `summary.json`,
and `eval_student.py` carries forward any phase recorded with the same eval config. Both student
phases decode greedily over identical problems, so the per-problem comparison is paired.

**Why the teacher is a phase.** The student cannot pass its teacher, so `after − before` alone is
uninterpretable. The reported metric is `headroom_recovered = (after − before) / (teacher − before)`.

In [ ]:
EVAL_STEPS = [25, 50]                      # every SAVE_EVERY checkpoint you want scored
STUDENT    = env_var("STUDENT_MODEL")
EVAL_ROOT  = env_var("EVAL_DIR")

# before (chosen student) + teacher from Part 1, merged into one summary to carry forward
base = json.loads(fsx_cat(f"{baseline_dir(STUDENT)}/summary.json"))
base["results"]["teacher"] = json.loads(fsx_cat(f"{baseline_dir(TEACHER_ID)}/summary.json"))["results"]["teacher"]

for step in EVAL_STEPS:
    ckpt, out = f"{RUN_DIR}/student-step{step}", f"{EVAL_ROOT}/step{step}"
    if not fsx_cat(f"{ckpt}/adapter_config.json").strip():
        print(f"step {step}: {ckpt} not written yet -- re-run this cell later"); continue
    fsx_write(f"{out}/summary.json", json.dumps(base, indent=2))
    run_eval(EVAL_PHASE="after", EVAL_DIR=out, EVAL_FT_MODEL_DIR=ckpt)

render("opd-config.yaml-template")         # leave the ConfigMap as env_vars defines it

In [ ]:
def print_eval(payload):
    cfg = payload["config"]
    print(f"task {payload['task']} · {cfg['dataset']} · {cfg['select']} {cfg['limit']} samples · "
          f"{cfg['fewshot']}-shot · temperature {cfg['temperature']}\n")
    phases = [p for p in ("before", "after", "teacher") if p in payload["results"]]
    hdr = f"{'metric':<22}" + "".join(f"{p:>12}" for p in phases)
    print(hdr); print("-" * len(hdr))
    for metric, fmt in [("accuracy", "{:.4f}"), ("stderr", "{:.4f}"), ("format_rate", "{:.2f}"),
                        ("parse_rate", "{:.2f}"), ("mean_output_tokens", "{:.1f}")]:
        cells_ = "".join(f"{fmt.format(r[metric]):>12}" if r.get(metric) is not None else f"{'-':>12}"
                         for r in (payload["results"][p] for p in phases))
        print(f"{metric:<22}{cells_}")
    if "delta_accuracy" in payload:
        print(f"\ndelta accuracy = {payload['delta_accuracy']:+.4f}  ·  per-sample flips: {payload['transitions']}")
    if payload.get("headroom_recovered") is not None:
        print(f"teacher gap = {payload['teacher_gap']:+.4f}  ·  headroom recovered = {payload['headroom_recovered']:.1%}")


DASH_STEP = EVAL_STEPS[0]                  # which checkpoint the dashboard below shows
payload = load_eval(f"{EVAL_ROOT}/step{DASH_STEP}")
print_eval(payload) if payload else print("No results yet -- run the cell above first.")

### Dashboard — before vs. after vs. teacher

Six panels for one checkpoint: headline **accuracy** (± standard error) with the **teacher line**
as the real ceiling, **running accuracy** across the problems, the **paired per-problem outcome**
(what distillation *fixed* vs. *broke*), **answer-format behaviour**, **response length**, and — for
context — the training signal `teacher_kl` from `${RUN_DIR}/metrics_rank0.jsonl`.

Panel 1 is the one to read first. `after` moving up is necessary but not sufficient; what matters is
how much of the `before → teacher` distance it covered.

In [ ]:
import matplotlib.pyplot as plt

# grey baseline · AWS-orange distilled · dark navy teacher (the ceiling, not a competitor)
COLORS = {"before": "#9aa0a6", "after": "#ff9900", "teacher": "#232f3e"}
STUDENT_PHASES = ("before", "after")


def eval_dashboard(payload, run_dir=RUN_DIR):
    results, samples = payload["results"], payload.get("samples", {})
    phases = [p for p in ("before", "after", "teacher") if p in results]
    students = [p for p in phases if p in STUDENT_PHASES]
    fig, ax = plt.subplots(2, 3, figsize=(15, 7.5))
    cfg = payload["config"]
    fig.suptitle(f"On-policy distillation — {payload['task']} · {cfg['dataset']} "
                 f"({cfg['select']} {cfg['limit']}, {cfg['fewshot']}-shot, greedy)", fontsize=13)

    # 1 -- headline accuracy +- stderr, with the teacher drawn as the ceiling
    a = ax[0, 0]
    accs = [results[p]["accuracy"] for p in phases]
    errs = [results[p].get("stderr") or 0 for p in phases]
    bars = a.bar(phases, accs, yerr=errs, capsize=6,
                 color=[COLORS.get(p, "#4285f4") for p in phases])
    for bar, acc, err in zip(bars, accs, errs):
        a.text(bar.get_x() + bar.get_width() / 2, acc + err + 0.01, f"{acc:.1%}",
               ha="center", va="bottom", fontweight="bold")
    if "teacher" in results and results["teacher"]["accuracy"] is not None:
        a.axhline(results["teacher"]["accuracy"], ls="--", lw=1.2, color="#232f3e", alpha=0.7)
    title = f"{payload['task']} accuracy"
    if "delta_accuracy" in payload:
        title += f"  (Δ {payload['delta_accuracy']:+.1%}"
        if payload.get("headroom_recovered") is not None:
            title += f", {payload['headroom_recovered']:.0%} of headroom"
        title += ")"
    a.set_title(title)
    a.set_ylim(0, max(accs + [0.05]) * 1.35); a.set_ylabel("accuracy"); a.grid(axis="y", alpha=0.3)

    # 2 -- running accuracy over the held-out samples
    a = ax[0, 1]
    for p in phases:
        rows = samples.get(p, [])
        running = [sum(r["correct"] for r in rows[:i + 1]) / (i + 1) for i in range(len(rows))]
        a.plot(range(1, len(running) + 1), running, label=p, color=COLORS.get(p), linewidth=2)
    a.set_title("Running accuracy"); a.set_xlabel("sample"); a.set_ylabel("accuracy")
    a.legend(); a.grid(alpha=0.3)

    # 3 -- paired per-sample outcome: what actually moved, before vs. after
    a = ax[0, 2]
    tr = payload.get("transitions")
    if tr:
        keys = ["fixed", "broken", "both_correct", "both_wrong"]
        a.bar([k.replace("_", "\n") for k in keys], [tr[k] for k in keys],
              color=["#1e8e3e", "#d93025", "#4285f4", "#9aa0a6"])
        for i, k in enumerate(keys):
            a.text(i, tr[k], str(tr[k]), ha="center", va="bottom")
        a.set_title(f"Paired outcome (fixed {tr['fixed']} vs. broken {tr['broken']})")
        a.set_ylabel("samples"); a.grid(axis="y", alpha=0.3)
    else:
        a.set_title("Paired outcome -- needs both `before` and `after`"); a.axis("off")

    # 4 -- answer-format behaviour: does the student still write a final answer after training?
    a = ax[1, 0]
    width, xs = 0.8 / max(len(phases), 1), range(2)
    for i, p in enumerate(phases):
        vals = [results[p].get("format_rate") or 0, results[p].get("parse_rate") or 0]
        a.bar([x + i * width for x in xs], vals, width, label=p, color=COLORS.get(p))
    a.set_xticks([x + width * (len(phases) - 1) / 2 for x in xs])
    a.set_xticklabels(["writes answer marker", "answer parsed"])
    a.set_ylim(0, 1.1); a.set_title("Answer format compliance"); a.set_ylabel("fraction of samples")
    a.legend(fontsize=8); a.grid(axis="y", alpha=0.3)

    # 5 -- response length distribution
    a = ax[1, 1]
    data = [[r["output_tokens"] for r in samples.get(p, []) if r.get("output_tokens")] for p in phases]
    if any(data):
        bp = a.boxplot(data, patch_artist=True, showfliers=False)
        for patch, p in zip(bp["boxes"], phases):
            patch.set_facecolor(COLORS.get(p, "#4285f4")); patch.set_alpha(0.75)
        a.set_xticks(range(1, len(phases) + 1)); a.set_xticklabels(phases)
        means = [results[p].get("mean_output_tokens") for p in phases]
        a.set_title("Response length  (mean " + " → ".join(f"{m:.0f}" for m in means if m) + ")")
    a.set_ylabel("output tokens"); a.grid(axis="y", alpha=0.3)

    # 6 -- training signal for context: teacher KL over steps
    a = ax[1, 2]
    tpod = sh_out(f"{KCTL} get pods -l app=teacher-vllm "
                  "-o jsonpath='{.items[0].metadata.name}'").strip()
    raw = sh_out(f"{KCTL} exec {tpod} -c vllm -- cat {run_dir}/metrics_rank0.jsonl") if tpod else ""
    rows = [json.loads(l) for l in raw.splitlines() if l.strip()]
    if rows:
        a.plot([r["step"] for r in rows], [r["teacher_kl"] for r in rows], color="#232f3e")
        a.set_title("Training signal: teacher reverse KL"); a.set_xlabel("step")
        a.set_ylabel("mean reverse KL"); a.grid(alpha=0.3)
    else:
        a.set_title("teacher_kl unavailable (needs a pod mounting /fsx)"); a.axis("off")

    plt.tight_layout(); plt.show()


if payload:
    eval_dashboard(payload)
else:
    print("Run the eval cell above first.")

### Final comparison — every baseline and the distilled student

The Part 1 dashboard again, with each evaluated checkpoint added: where the distilled 2B lands
between its own base, the other candidate and the teacher, and how many teacher-solved problems it
still misses (276 for the base 2B → 159 after 25 steps).

In [ ]:
final = [phase_entry(baseline_dir(m), "before", f"{short(m)}\nbase",
                     "chosen" if m == STUDENT else "candidate") for m in CANDIDATES]
final += [phase_entry(f"{EVAL_ROOT}/step{s}", "after", f"{short(STUDENT)}\nstep {s}", "distilled")
          for s in EVAL_STEPS]
final.append(phase_entry(baseline_dir(TEACHER_ID), "teacher", f"{short(TEACHER_ID)}\nteacher", "teacher"))
compare_dashboard(final, f"{TASK}: baselines vs. the distilled {STUDENT.split('/')[-1]} (no SFT)")

## Results — GSM8K, Qwen3.5-9B → Qwen3.5-2B (LoRA r=128), no SFT

All 1,319 GSM8K test problems, greedy, 4-shot; *z* is the paired sign test against the base
student; "missed" counts problems the teacher solves and the model does not.

| model / checkpoint | accuracy | vs base | fixed / broken | gap to teacher closed | missed | answer marker | capped |
|---|---|---|---|---|---|---|---|
| Qwen3.5-0.8B base (candidate) | 52.0% | — | — | — | 567 | 94.1% | 6.1% |
| **Qwen3.5-2B base** | **74.8%** | — | — | — | 276 | 94.6% | 5.5% |
| **2B, step 25** | **84.0%** | **+9.2 (z = +7.45)** | **195 / 73** | **47.3%** | **159** | 93.7% | 6.4% |
| 2B, step 50 | 83.9% | +9.2 (z = +7.49) | 191 / 70 | 46.9% | 160 | 94.1% | 5.9% |
| Qwen3.5-9B teacher | 94.3% | — | — | — | — | 98.5% | 1.5% |

**How the recipe got here** (every run: `experiments/README.md`, every checkpoint:
`experiments/results/gsm8k_evals.csv`):

| run | student | change | best |
|---|---|---|---|
| run13 | 0.8B | first working recipe: LR 1e-5, 100 steps | 64.7% (30% of the gap) |
| run14–15 | 0.8B | 200 steps; LR 2e-5 | 65.8%, 65.1% — the same plateau, reached sooner at 2e-5 |
| run16 | 2B | larger student, LoRA r=32 | 83.2% (43%) |
| **run17** | **2B** | **LoRA r=128 — the shipped configuration** | **84.0% (47%) at step 25** |
| run18 | 2B | temperature 1.0, 8 samples × 32 prompts | 84.4% at step 50 — a tie within noise (±1.3 pp) |

**What made it work** (each found by a failed run):

- **A moderate LR.** At 1e-4 the student collapsed into long non-answers within ~4 steps; 1e-5 and
  2e-5 are both stable, and 2e-5 reaches the plateau ~4× sooner.
- **`ANSWER_SPAN_POLICY=mask`.** The final-answer tokens carry the largest reverse KL (the teacher can
  tell when the number is wrong), so without masking the student learns to stop writing answers.
- **`WHITEN_ADVANTAGES=1`.** The rollouts' reverse KL averages ~+0.3, so without a baseline nearly every
  sampled token is pushed down and the student's distribution flattens.
- **The adapter-renaming fix** in `sampler.py`, without which the sampler never applies the trained LoRA.
- **Choosing the student from the baseline.** The limit is the student's capacity, not training
  length or LR: the 0.8B plateaus at ~65% whatever the schedule, the 2B gains 9 points in 25 steps.

Thinking mode does not help here (0.8B 52.8% → 37.4% on 500 problems: 78% of answers hit the
3,500-token cap). Experimental, off by default: `TOPK_KL=20` adds a KL to the teacher's top-20
next-token distribution at every position (run19).

## Part 5 — Cleanup

Delete the Jobs, the sampler, the teacher `Service` and the ConfigMap to free the node.

**Order matters for one of them:** the teacher Deployment holds `TEACHER_TP` (2) GPUs for as long
as it exists, but the eval's `teacher` phase needs it (and `/fsx` is read through it) — so delete the
teacher *last*, after the evals and dashboards are done. To free a GPU mid-run without touching the
teacher, delete `student-sampler` instead. Baseline and eval results stay on `/fsx`.

In [ ]:
sh(f"{KCTL} delete job opd-student --ignore-not-found")
sh(f"{KCTL} delete job opd-student-eval --ignore-not-found")
sh(f"{KCTL} delete deploy,svc student-sampler --ignore-not-found")
sh(f"{KCTL} delete deploy,svc teacher-vllm --ignore-not-found")    # last: the eval needs it
sh(f"{KCTL} delete configmap opd-config --ignore-not-found")
sh(f"{KCTL} get pods -o wide")